# Spark X2.6 4B Recursive72 — Statistical Stratified-Sample Benchmark Suite

**Model:** `TassP0lar/Spark-X2.6-4B-Recursive72` (atop frozen `XHToken/Spark-X2.6-4B`)  
**Architecture:** 36 physical layers $\rightarrow$ 72 virtual visits (`4 + 24 + 24 + 12 + 8` = 18 global + 54 local), bounded-gain `TwoStreamMHC` Schema v2 (`10,078,176` params), and 18-visit Stage E `VisitAwareDLSAIndexBank` (`1,769,472` selector params).

### Why This Notebook Exists (Alongside `SparkX_Loop72_DLSA_Benchmark_Colab.ipynb`)
Running 100% of every long-context and reasoning benchmark at `max_new_tokens=32768` takes days on a single GPU. This notebook uses **survey-sampling statistics** (`sparkx_stat_benchmark.py`) to obtain **unbiased, uncertainty-quantified accuracy estimates in ~45 minutes (3 long-context suites) to ~1.8–2.8 hours (all 8 core suites)**:

1. **Zero Prefix/Order Bias (Stratified Random Sampling Without Replacement):** Never takes `rows[:n]` (which overestimates contest benchmarks like AIME ordered easy $\rightarrow$ hard, or skews domain/length mixes in LongBench v2 & GraphWalks). Candidate rows are validated (`status == 'ready'`) and proportionally sampled across difficulty/domain/length strata using a deterministic SHA-256 seed.
2. **Finite Population Correction ($\text{FPC} = \sqrt{(N - n)/(N - 1)}$):** Automatically shrinks standard errors when sampling a substantial fraction of finite splits (`AIME 2026` $N=30$, `HMMT Feb 2026` $N=30$, `GPQA Diamond` $N=198$, `LongBench v2` $N=503$) — reaching **exact zero sampling error ($\text{FPC}=0$)** when $n = N$!
3. **Exact Uncertainty Quantification:** Reports FPC-adjusted **Wilson Score 95% CIs** for binary `{0, 100}` tasks and **Rao-Wu Finite-Population Rescaled Bootstrap 95% CIs** (`B=10,000`) for continuous `[0, 100]` tasks (`GraphWalks` F1, `MRCR` sequence match, `IFEval` constraint score), plus **Neyman-Cochran stratified SE**.
4. **Crash-Safe Per-Question Resume:** Pins `<suite>.sample_manifest.json` and flushes SHA-256-verified `<suite>.stat.jsonl` after every single question (`resume=True`).
5. **Strict Benchmark Honesty:** Separates Tier-1 `official`-adapter long-context subsets (`graphwalks_bfs_128k`, `mrcr_8needle`, `longbench_v2`) from Tier-2 `diagnostic` real-HF verifiers (`aime_2026`, `hmmt_feb_2026`, `ifeval`, `gpqa_diamond`, `bfcl_v4_simple`, `imo_answerbench`, `gaokao_2026`), always recording `sampling_mode='statistical_subset'` and `official_comparable=False`.


In [ ]:
import importlib, json, os, subprocess, sys, time
from pathlib import Path

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
BRANCH = 'loop72-dlsa'
REPO_DIR = '/content/Spark-X2.6-Recursion' if os.path.exists('/content') else os.getcwd()

if os.path.exists('/content'):
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    if os.environ.get('SPARKX_SKIP_GIT_SYNC') != '1':
        subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-omp.txt'], check=True)
    importlib.invalidate_caches()
    for _m in (
        'sparkx_dlsa', 'sparkx_paged_cache', 'sparkx_sparse_attention', 'sparkx_loop_v03_core',
        'load_inference_v03', 'sparkx_loop72_dlsa_benchmarks', 'sparkx_full_benchmark', 'sparkx_stat_benchmark',
    ):
        if _m in sys.modules:
            importlib.reload(sys.modules[_m])

GIT_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
import torch
import load_inference_v03 as loader
import sparkx_full_benchmark as full
import sparkx_loop72_dlsa_benchmarks as bench
import sparkx_stat_benchmark as stat
from train_spark_mhc import DEFAULT_HF_REPO_ID, normalize_hf_repo_id, resolve_hf_token

print(f'Synced {REPO_URL} @ {BRANCH} ({GIT_COMMIT[:8]}) | PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}')


In [ ]:
# === 1. Configure Statistical Budget Preset & Sampling Parameters ===
# Presets across all 8 core suites (@27.4 tok/s): 'ultra_fast' (~1.8 hr; ~45 min if only 3 long-context suites) |
#   'quick' (~2.8 hr; ~68 min if only 3 long-context suites) | 'standard' (~6.1 hr, full 30/30 AIME & HMMT) | 'thorough' (~9.8 hr)
PRESET = 'quick'
SEED = 20261110
CONFIDENCE = 0.95
MAX_CONTEXT_TOKENS = 131072  # Keep single-GPU long-context prompts <= 128K tokens (set to 262144 or None for wider horizon)
SAMPLING_POLICY = 'greedy'   # 'greedy' (deterministic temp=0) or 'official_spark_x26' (temp=1.0, top_p=0.95)

# Core suites (uncomment 'imo_answerbench' or 'gaokao_2026' to include additional math suites):
SUITES = [
    'graphwalks_bfs_128k',  # Tier 1: Official-adapter Long Context (pinned @be6cc6ec)
    'mrcr_8needle',         # Tier 1: Official-adapter Long Context (pinned @f4c69fae)
    'longbench_v2',         # Tier 1: Official-adapter Long Context (pinned @2b48e494, N=503)
    'aime_2026',            # Tier 2: Diagnostic Math (N=30)
    'hmmt_feb_2026',        # Tier 2: Diagnostic Math (N=30)
    'ifeval',               # Tier 2: Diagnostic Instruction Following
    'gpqa_diamond',         # Tier 2: Diagnostic Science MCQ (N=198; skipped gracefully if HF_TOKEN lacks GPQA gate)
    'bfcl_v4_simple',       # Tier 2: Diagnostic Tool-Calling AST
    # 'imo_answerbench',
    # 'gaokao_2026',
]

# Optional per-suite sample-size overrides, e.g. {'aime_2026': 30} to run full AIME even in 'quick' mode:
SAMPLE_SIZE_OVERRIDES = {}
MAX_NEW_TOKENS_OVERRIDES = {}

# Deterministic output directory so re-running after a Colab disconnect automatically resumes from the exact unfinished item!
SAVE_TO_GOOGLE_DRIVE = False
if SAVE_TO_GOOGLE_DRIVE and os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_OUT_ROOT = Path('/content/drive/MyDrive/SparkX_Recursive72_StatBench')
else:
    BASE_OUT_ROOT = Path('runs')

RUN_DIR = BASE_OUT_ROOT / f'stat_eval_{PRESET}_{SAMPLING_POLICY}_ctx{MAX_CONTEXT_TOKENS or "all"}_seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

BUDGET_ESTIMATE = stat.estimate_preset_budget_table(PRESET, suites=SUITES, decode_tok_per_s=27.4, confidence=CONFIDENCE)
print(f'Output directory (resumable): {RUN_DIR}')
print(f"Planned questions: {BUDGET_ESTIMATE['total_planned_questions']} | Estimated GPU time (@27.4 tok/s): ~{BUDGET_ESTIMATE['estimated_total_minutes']} min (~{BUDGET_ESTIMATE['estimated_total_hours']} hr)\n")
print(f"{'Suite':<22} {'Mode':<11} {'n':>4} {'~N':>6} {'FPC':>6} {'Exp ±95%CI':>11} {'MaxNewTok':>10} {'~Minutes':>9}")
print('-' * 85)
for r in BUDGET_ESTIMATE['suites']:
    n_eff = SAMPLE_SIZE_OVERRIDES.get(r['suite'], r['planned_n'])
    pop_str = str(r['approx_population_N']) if r['approx_population_N'] is not None else 'stream'
    print(f"{r['suite']:<22} {r['task_mode']:<11} {n_eff:>4} {pop_str:>6} {r['fpc_factor']:>6.2f} {'±' + str(r['expected_95ci_half_width_pct']) + '%':>11} {r['max_new_tokens']:>10} {r['est_minutes']:>8.1f}m")


In [ ]:
# === 2. Load Validated 72-Layer Model (`TassP0lar/Spark-X2.6-4B-Recursive72`) ===
HF_TOKEN = resolve_hf_token()
LOCAL_EXPORT_DIR = Path('results/mhc_colab_g4_run/final_loop72_mhc_dlsa_export')
LOCAL_SELECTOR_CKPT = Path('results/mhc_colab_g4_run/visit_selectors_stage_e.pt')

if LOCAL_EXPORT_DIR.is_dir():
    CHECKPOINT_DIR = str(LOCAL_EXPORT_DIR)
    DLSA_CHECKPOINT = str(LOCAL_SELECTOR_CKPT) if LOCAL_SELECTOR_CKPT.is_file() else str(LOCAL_EXPORT_DIR / 'visit_selectors_stage_e.pt')
else:
    CHECKPOINT_DIR = normalize_hf_repo_id(os.environ.get('HF_REPO_ID', DEFAULT_HF_REPO_ID), token=HF_TOKEN)
    DLSA_CHECKPOINT = 'visit_selectors_stage_e.pt'

MODEL_REVISION = full.MODEL_REVISION
VARIANT_REVISION = os.environ.get('VARIANT_REVISION') or None
print(f'Loading Spark X2.6 4B Recursive72 from: {CHECKPOINT_DIR}')

model, tokenizer = loader.load_sparkx_loop72_lsa(
    repo_or_dir=CHECKPOINT_DIR,
    base_repo='XHToken/Spark-X2.6-4B',
    base_revision=MODEL_REVISION,
    variant_revision=VARIANT_REVISION,
    dlsa_checkpoint=DLSA_CHECKPOINT,
    recurrence_mode='learned_mhc',
    strict_selector_provenance=True,
    require_verified_selectors=True,
    token=HF_TOKEN,
)

manifest = model.execution_manifest()
assert manifest['total_virtual_layers'] == 72
assert manifest['global_visits'] == 18 and manifest['local_visits'] == 54
assert manifest['effective_recurrence_mode'] == 'learned_mhc'
assert manifest['mhc_gain_mode'] == 'sigmoid' and manifest['mhc_schema_version'] == 2
assert manifest['selector_checkpoint_status']['initialization'] in ('checkpoint_loaded_verified', 'loop_state_checkpoint_loaded_verified')
assert manifest['selector_checkpoint_status']['trained_global_visits'] == 18
(RUN_DIR / 'model_manifest.json').write_text(json.dumps(manifest, indent=2))

# Quick 128-token calibration probe to measure this GPU's exact decode tok/s:
CAL_SPEED = bench.benchmark_loop72_decode_speed(
    model, tokenizer, prompt_lengths=(512, 8192), gen_tokens=128, warmups=1, repeats=1, verbose=True,
)
measured_tps = float(CAL_SPEED['horizons']['512']['decode_tok_per_s_median'])
UPDATED_EST = stat.estimate_preset_budget_table(PRESET, suites=SUITES, decode_tok_per_s=measured_tps, confidence=CONFIDENCE)
print(f"\nMeasured GPU decode speed: {measured_tps:.1f} tok/s -> Calibrated preset wall time: ~{UPDATED_EST['estimated_total_minutes']} min (~{UPDATED_EST['estimated_total_hours']} hr)")


In [ ]:
# === 3. Run Stratified Statistical Benchmark Suite (Crash-Safe & Resumable) ===
if SAMPLING_POLICY == 'official_spark_x26':
    GEN_OPTIONS = {
        'do_sample': True,
        'temperature': 1.0,
        'top_p': 0.95,
        'top_k': -1,
        'repetition_penalty': 1.0,
        'force_think_close': False,
    }
else:
    GEN_OPTIONS = {
        'do_sample': False,
        'temperature': 0.0,
        'top_p': 1.0,
        'top_k': -1,
        'repetition_penalty': 1.0,
        'force_think_close': False,
    }

STAT_REPORT = stat.run_statistical_benchmarks(
    model=model,
    tokenizer=tokenizer,
    output_dir=RUN_DIR,
    preset=PRESET,
    suites=SUITES,
    sample_size_overrides=SAMPLE_SIZE_OVERRIDES,
    max_new_tokens_overrides=MAX_NEW_TOKENS_OVERRIDES,
    max_context_tokens=MAX_CONTEXT_TOKENS,
    seed=SEED,
    confidence=CONFIDENCE,
    generation_options=GEN_OPTIONS,
    token=HF_TOKEN,
    verbose=True,
    resume=True,
)

try:
    from IPython.display import Markdown, display
    display(Markdown('## Statistical Benchmark Summary (`95% CI` with Finite Population Correction)\n\n' + STAT_REPORT['markdown_table']))
except Exception:
    pass


In [ ]:
# === 4. Optional: Paired Common-Random-Numbers (CRN) 72L vs 36L Base Test ===
# Evaluates 72L Recursive + mHC + D-LSA and 36L Frozen Base on the EXACT SAME sampled questions
# (reusing the already-loaded 4B backbone weights in-place with zero extra VRAM!), cancelling item-difficulty
# variance and computing the paired bootstrap 95% CI + exact sign-flip permutation p-value on (72L - 36L).
RUN_PAIRED_ABLATION = False
PAIRED_SUITE = 'aime_2026'  # e.g. 'aime_2026', 'ifeval', 'graphwalks_bfs_128k', 'bfcl_v4_simple'
PAIRED_N = 12

if RUN_PAIRED_ABLATION:
    PAIRED_OUT = stat.run_paired_ablation_suite(
        model=model,
        tokenizer=tokenizer,
        suite_name=PAIRED_SUITE,
        output_dir=RUN_DIR / f'paired_{PAIRED_SUITE}_n{PAIRED_N}',
        sample_size=PAIRED_N,
        seed=SEED,
        max_context_tokens=min(65536, MAX_CONTEXT_TOKENS or 65536),
        confidence=CONFIDENCE,
        generation_options=GEN_OPTIONS,
        token=HF_TOKEN,
        verbose=True,
        resume=True,
    )
    print('\n=== Paired Common-Random-Numbers Comparison (72L Recursive vs 36L Frozen Base) ===')
    print(json.dumps(PAIRED_OUT['paired_comparison'], indent=2))
